# GRM shard timing / feasibility check

Before committing to a full sharded GRM run (`plink --make-grm-bin --parallel k n`
across `n` shards), this times a handful of representative shards for a
candidate `n` (200, per the initial sizing conversation), checks whether
PLINK's row-balancing actually keeps per-shard cost uniform across `k`
(theory says it should -- the split balances off-diagonal *pair* counts, not
raw row counts -- but this repo's convention throughout is to measure, not
assume: `chr22_qc_thinning_timing.ipynb` timed one chromosome before the
genome-wide QC run, `THIN_P` was calibrated empirically), and extrapolates
total single-threaded compute plus a feasibility table of wall-clock
estimates at a few concurrency levels.

**Not a production run.** This doesn't compute the full GRM -- it's the same
role `chr22_qc_thinning_timing.ipynb` played for QC: a calibration step whose
numbers feed into the real sharded-GRM notebook (not yet built) once a shard
count is actually decided.

## Compute resource

Same `n1-highmem-64` (64 vCPU / 416 GB) as the QC + merge step. PLINK 1.9's
`--make-grm-bin` is single-threaded (no useful `--threads` lever), and every
shard -- however few rows it covers -- still loads the **entire** packed
genotype panel into memory, since any row's relatedness needs every other
sample's genotypes too. So concurrency is bounded by RAM ÷ (per-shard
memory) *and* by vCPU count (each concurrent shard is still its own
process competing for a core) -- whichever is smaller. This notebook
measures actual peak memory per shard (polling `/proc/<pid>/status`'s
`VmHWM`, no root or extra tooling needed) rather than estimating it from
the packed file size, since plink's real working-set can be a multiple of
the raw packed data; it takes the vCPU count from `os.cpu_count()` rather
than assuming RAM is always the binding constraint.

## Setup

PLINK 1.9, not plink2 -- `GRM-pairs/grm_bin_sharded`'s row-range recovery
(used later, for the phenotype cross-product step) is calibrated to PLINK
1.9's own `--parallel` split algorithm specifically. This notebook is plink
only -- no `GRM-pairs` build or code path touched here.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/1.9/
  PLINK_URL="https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20230116.zip"
  cd /tmp
  wget -q -O plink.zip "$PLINK_URL"
  unzip -o -q plink.zip plink -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink"
fi

export PATH="$BIN_DIR:$PATH"
plink --version

nproc
free -h

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Paths

`BED_PREFIX`: the genome-wide PLINK1 bed/bim/fam `genome_wide_qc_thinning_merge.ipynb`
persisted to the bucket. Copied to local scratch first, same convention as
everywhere else in this pipeline -- plink reading a multi-GB bed repeatedly
over the gcsfuse-mounted bucket is much slower than local disk.

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

BED_NAME = "genome_wide_round2b_thinned_bed"   # from genome_wide_qc_thinning_merge.ipynb
BED_PREFIX = os.path.join(LOCAL_WORK_DIR, BED_NAME)

for ext in ("bed", "bim", "fam"):
    bucket_path = f"{BUCKET_DIR}/{BED_NAME}.{ext}"
    local_path = f"{BED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), f"missing merged bed export: {bucket_path!r} -- run genome_wide_qc_thinning_merge.ipynb's merge section first"
    if not os.path.isfile(local_path):
        import shutil
        shutil.copy(bucket_path, local_path)

print(BED_PREFIX)

In [ ]:
%%bash -s "$BED_PREFIX"
set -e
BED_PREFIX=$1

echo "N (samples):"
wc -l < "${BED_PREFIX}.fam"
echo "M (variants):"
wc -l < "${BED_PREFIX}.bim"
echo "bed file size:"
ls -lh "${BED_PREFIX}.bed" | awk '{print $5}'

## Run parameters

`N_SHARDS`: the candidate to check, not yet a decision. `SAMPLE_KS`: a
handful of shards spread across the *full* range (first, ~25%, ~50%, ~75%,
last), not just one -- if PLINK's row-balancing is working as documented,
these should all take roughly the same time despite covering different row
ranges; if they don't, that's worth knowing before assuming uniform cost
lets you just multiply one timing by `N_SHARDS`. Keep these two in sync --
`SAMPLE_KS` clustered at one end of a much larger `N_SHARDS` isn't
representative of the whole range.

In [ ]:
N_SHARDS = 2000
SAMPLE_KS = [1, 500, 1000, 1500, 2000]   # spread across the full range, not clustered at one end

## Precompute allele frequencies + cap plink's memory reservation

Two things learned from a real timed shard (`k=1 of 2000`, 13 min):

1. **Every shard sees the identical sample/variant panel** (no per-shard
   `--keep`), so `--make-grm-bin`'s own "Calculating allele frequencies..."
   step recomputes the exact same numbers on every one of `N_SHARDS` runs --
   pure redundant work. Precomputed once here via `--freq`, reused via
   `--read-freq` in every shard call below.
2. **PLINK 1.9 defaults to reserving half of detected RAM** as its working
   space regardless of actual need (the real log showed `"418995 MB RAM
   detected; reserving 209497 MB for main workspace"`) -- if that
   reservation gets touched, every shard's measured peak RSS reflects that
   default, not its true working set, which would cap RAM-bounded
   concurrency at just 1-2 shards on this machine regardless of `N_SHARDS`.
   `--memory` below caps it to roughly 2x the bed file size plus headroom
   instead, sized from the actual file, not guessed.

In [ ]:
%%bash -s "$BED_PREFIX"
set -e
BED_PREFIX=$1

plink --bfile "$BED_PREFIX" --freq --out "${BED_PREFIX}_freq"

In [ ]:
FREQ_PATH = f"{BED_PREFIX}_freq.frq"
assert os.path.isfile(FREQ_PATH), f"missing precomputed frequencies: {FREQ_PATH!r}"

BED_SIZE_GB = os.path.getsize(f"{BED_PREFIX}.bed") / 1e9
MEMORY_MB = int((BED_SIZE_GB * 2 + 4) * 1024)   # 2x bed size + 4 GB headroom, sized from
                                                 # the actual file rather than plink's own
                                                 # half-of-detected-RAM default
print(f"FREQ_PATH = {FREQ_PATH}")
print(f"Bed file size: {BED_SIZE_GB:.1f} GB -> --memory {MEMORY_MB} MB")

## Driver

One `--parallel k N_SHARDS --make-grm-bin` call per `k` in `SAMPLE_KS`, now
with `--read-freq FREQ_PATH` (skip redundant per-shard frequency
recomputation) and `--memory MEMORY_MB` (cap plink's default half-of-RAM
reservation to something sized from the actual bed file). Wall-clock via
Python's own `time.monotonic()` (same convention as
`genome_wide_qc_thinning_merge.ipynb`'s driver); peak memory via a
background thread polling `/proc/<pid>/status`'s `VmHWM` ("high water
mark" -- the kernel's own running peak-RSS tracker for that process) while
plink runs, rather than `/usr/bin/time -v` -- that needs installing via
`apt`, which needs root, which this environment's `jupyter` user doesn't
have. This needs no privileges and no external tool at all. Plink only --
no `GRM-pairs`/`grm_shard_tool` involved at this stage.

In [ ]:
import subprocess
import threading
import time

def get_n_ids(bed_prefix):
    with open(f"{bed_prefix}.fam") as f:
        return sum(1 for _ in f)

N_IDS = get_n_ids(BED_PREFIX)
print(f"N_IDS = {N_IDS}")

def peak_rss_kb(pid, samples, poll_interval=0.2, stop_event=None):
    # VmHWM is the kernel's own running peak-RSS ("high water mark") for this pid --
    # polling it just samples an already-monotonically-increasing value, so a modest
    # poll interval doesn't risk missing a brief spike the way sampling raw RSS would
    status_path = f"/proc/{pid}/status"
    while not (stop_event and stop_event.is_set()):
        try:
            with open(status_path) as f:
                for line in f:
                    if line.startswith("VmHWM:"):
                        samples.append(int(line.split()[1]))
                        break
        except FileNotFoundError:
            break   # process already exited
        time.sleep(poll_interval)

def time_shard(k, n_shards):
    out_prefix = os.path.join(LOCAL_WORK_DIR, f"grm_shard_{k}_of_{n_shards}")
    cmd = ["plink", "--bfile", BED_PREFIX, "--make-grm-bin",
           "--parallel", str(k), str(n_shards),
           "--read-freq", FREQ_PATH, "--memory", str(MEMORY_MB),
           "--out", out_prefix]

    rss_samples = []
    stop_event = threading.Event()

    start = time.monotonic()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    monitor = threading.Thread(target=peak_rss_kb, args=(proc.pid, rss_samples), kwargs={"stop_event": stop_event})
    monitor.start()
    stdout, _ = proc.communicate()
    stop_event.set()
    monitor.join(timeout=1)
    elapsed_sec = time.monotonic() - start

    max_rss_kb = max(rss_samples) if rss_samples else None
    bin_size_bytes = os.path.getsize(f"{out_prefix}.grm.bin") if os.path.isfile(f"{out_prefix}.grm.bin") else None

    return {
        "k": k, "n_shards": n_shards,
        "elapsed_sec": elapsed_sec, "max_rss_gb": max_rss_kb / 1e6 if max_rss_kb else None,
        "bin_size_mb": bin_size_bytes / 1e6 if bin_size_bytes else None,
        "returncode": proc.returncode, "stdout_tail": stdout[-500:] if proc.returncode != 0 else None,
    }

shard_timings = [time_shard(k, N_SHARDS) for k in SAMPLE_KS]
for r in shard_timings:
    # max_rss_gb can still come back None if the process exited faster than the first
    # poll -- format defensively instead of crashing on a NoneType
    status = "ok" if r["returncode"] == 0 else f"FAILED (returncode={r['returncode']}, see stdout_tail)"
    elapsed = f"{r['elapsed_sec']:.1f}s" if r["elapsed_sec"] is not None else "N/A"
    rss = f"{r['max_rss_gb']:.2f} GB RSS" if r["max_rss_gb"] is not None else "N/A"
    print(f"k={r['k']:>3}: {elapsed}, {rss}, {status}")

## Feasibility summary

`elapsed_sec` across `SAMPLE_KS` should be roughly flat if plink's
row-balancing is working as documented -- a large spread means shard cost
isn't uniform and picking `N_SHARDS` needs more care than just dividing
total time by `n`. `max_rss_gb` is the real per-shard memory footprint,
measured, not estimated -- `MemAvailable` (`/proc/meminfo`, read live rather
than a hardcoded nominal spec number) ÷ `max_rss_gb` is how many shards can
actually run concurrently on this machine, which is the number that
matters for wall-clock, not `N_CONCURRENT` guessed from vCPU count the way
the QC step did. `MemAvailable` already accounts for reclaimable
buffer/cache (per `free -h`'s `available` column), so it needs less of an
extra safety margin than a raw total would.

In [ ]:
import pandas as pd

def get_available_ram_gb():
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemAvailable:"):
                return int(line.split()[1]) / 1e6   # kB -> GB
    return None

timing_df = pd.DataFrame(shard_timings)
display(timing_df[["k", "elapsed_sec", "max_rss_gb", "bin_size_mb"]])

avg_shard_sec = timing_df["elapsed_sec"].mean()
spread_pct = 100 * (timing_df["elapsed_sec"].max() - timing_df["elapsed_sec"].min()) / avg_shard_sec
max_rss_gb = timing_df["max_rss_gb"].max()

AVAILABLE_RAM_GB = get_available_ram_gb()   # measured live, not a hardcoded nominal spec number
RAM_SAFETY_FACTOR = 0.9   # MemAvailable already accounts for reclaimable cache, so less margin needed than a raw total

ram_bound_concurrent = int((AVAILABLE_RAM_GB * RAM_SAFETY_FACTOR) // max_rss_gb)

# --make-grm-bin is documented single-threaded, but the real plink log still printed
# "Using up to 63 threads" -- concurrency also needs a vCPU cap, not just RAM, or running
# many shards at once just serializes on CPU instead of RAM (which the RAM-only figure
# would silently hide, since RSS doesn't reflect CPU contention). Leave a couple cores
# for the OS/notebook kernel/monitor threads rather than claiming every one of them.
N_VCPUS = os.cpu_count()
CPU_HEADROOM = 2
cpu_bound_concurrent = max(1, N_VCPUS - CPU_HEADROOM)

max_concurrent = min(ram_bound_concurrent, cpu_bound_concurrent)
binding_constraint = "RAM" if ram_bound_concurrent <= cpu_bound_concurrent else "CPU"

print(f"Available RAM (measured, /proc/meminfo): {AVAILABLE_RAM_GB:.1f} GB")
print(f"Average per-shard time: {avg_shard_sec:.1f} sec")
print(f"Spread across sampled k: {spread_pct:.1f}% (large -> row-balancing isn't as uniform as assumed)")
print(f"Peak memory per shard: {max_rss_gb:.1f} GB")
print(f"RAM-bounded max concurrent shards: {ram_bound_concurrent}")
print(f"CPU-bounded max concurrent shards ({N_VCPUS} vCPUs, {CPU_HEADROOM} reserved): {cpu_bound_concurrent}")
print(f"Actual max concurrent shards on this machine: {max_concurrent} (binding constraint: {binding_constraint})")
print()

total_compute_hours = avg_shard_sec * N_SHARDS / 3600
print(f"Extrapolated total single-threaded compute for N_SHARDS={N_SHARDS}: {total_compute_hours:.1f} hours")

feasibility = pd.DataFrame({
    "n_concurrent": [1, max(1, max_concurrent // 2), max_concurrent],
}).assign(
    wall_clock_hours=lambda d: (N_SHARDS * avg_shard_sec / d["n_concurrent"]) / 3600
)
feasibility

## Next steps

If `spread_pct` is small and `wall_clock_hours` at `max_concurrent` looks
reasonable, `N_SHARDS` (whatever was tested) is a reasonable choice to
commit to for the real sharded-GRM notebook -- run all `N_SHARDS` shards at
`max_concurrent` concurrency (same `ThreadPoolExecutor`-managed
`subprocess` pattern as `genome_wide_qc_thinning_merge.ipynb`'s driver, now
capped by `min(RAM-bound, CPU-bound)` rather than either alone). See
`grm_shard_run.ipynb` for the production run built from these numbers.

Only one `.grm.id` needs to be kept out of the whole run: every shard sees
the identical sample panel (no per-shard `--keep`), so every shard's
`.grm.id` is byte-identical -- keep one (e.g. shard `k=1`'s), skip
persisting the other `N_SHARDS - 1` copies.

That notebook is plink only, same as this one -- `GRM-pairs`/`grm_shard_tool`
(`accumulate`/`merge`) comes in later, as a separate step, for the
phenotype cross-product against the finished GRM shards (where GRM-side
and phenotype-side `(FID, IID)` alignment matters -- see
`03_grm_shards/README.md`'s note on that). If wall-clock still looks too
long even at `max_concurrent`, the next lever is `dsub`/Google Batch
(separate machines, each with their own RAM *and* CPU budget) rather than
pushing concurrency higher on one machine.